# 03 — Baseline Model
A non-deep baseline (Random Forest on hand-crafted spectral-index + sensor features) to sanity-check the problem and set a performance floor before the CNN/LSTM models.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

from dataset import load_processed_field
from features import build_feature_table

PROCESSED_DIR = "../data/processed"
FIELD_ID = sorted(os.listdir(PROCESSED_DIR))[0]


## Load processed patches, build hand-crafted features (spectral index mean/std per patch)

In [ ]:
dates, patches_by_date, sensors, coords = load_processed_field(FIELD_ID, PROCESSED_DIR)

# use the most recent date's patches as the feature snapshot for this baseline
X_spectral, spectral_cols = build_feature_table(patches_by_date[-1])

# broadcast that date's sensor reading across all patch locations
sensor_row = sensors[-1] if sensors is not None else np.zeros(8, dtype=np.float32)
X_sensor = np.tile(sensor_row, (len(X_spectral), 1))
sensor_cols = [f"sensor_{i}" for i in range(X_sensor.shape[1])]

X = np.concatenate([X_spectral, X_sensor], axis=1)
feature_names = spectral_cols + sensor_cols
print("X shape:", X.shape)


## Labels (broadcast field-level label to every patch, same as 02_preprocessing.ipynb)

In [ ]:
labels_df = pd.read_csv(f"../data/labels/{FIELD_ID}.csv", parse_dates=["timestamp"])
y_row = labels_df[["stress_risk", "water_risk", "pest_risk"]].iloc[-1].values.astype(np.float32)
y = np.tile(y_row, (len(X), 1))
print("y shape:", y.shape)


## Train / test split + Random Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = MultiOutputRegressor(
    RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
)
baseline.fit(X_train, y_train)


## Evaluate

In [ ]:
y_pred = baseline.predict(X_test)
target_names = ["stress_risk", "water_risk", "pest_risk"]

for i, name in enumerate(target_names):
    mae = mean_absolute_error(y_test[:, i], y_pred[:, i])
    r2 = r2_score(y_test[:, i], y_pred[:, i])
    print(f"{name:12s} MAE={mae:.4f}  R2={r2:.4f}")


## Feature importance (averaged across the 3 targets)

In [ ]:
importances = np.mean([est.feature_importances_ for est in baseline.estimators_], axis=0)
order = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
plt.barh([feature_names[i] for i in order[:15]][::-1], importances[order[:15]][::-1])
plt.title("Top 15 feature importances (RandomForest baseline)")
plt.tight_layout()
plt.show()


### Notes
This baseline ignores temporal trends and treats every patch independently with a single snapshot in time. The CNN (`04_cnn.ipynb`) and CNN+LSTM (`05_cnn_lstm.ipynb`) models should beat these MAE/R2 numbers — if they don't, revisit label quality or feature alignment before adding model complexity.